In [1]:
from Crypto.PublicKey import RSA
from Crypto.Cipher import AES, PKCS1_OAEP
import socket

HOST = "127.0.0.1"
PORT = 40005

private_key = RSA.import_key(open("private.pem").read())

if __name__ == "__main__":
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

    server.bind((HOST, PORT))
    server.listen(0)

    connection, client_address = server.accept()

    # Imamo connection, zdaj preberemo potrebno za dekripcijo

    enc_session_key = connection.recv(private_key.size_in_bytes())

    # Imamo vse za dekripcijo, dešifrirajmo session_key

    cipher_rsa = PKCS1_OAEP.new(private_key)
    session_key = cipher_rsa.decrypt(enc_session_key)
    print('Session key is ', session_key)

#     cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)

    while True:
        # V zanki sprejemamo sporočila in jih vse dešifriramo s session_key, ki ga zdaj imamo
        
        nonce = connection.recv(16)
        tag = connection.recv(16)
        print(nonce)
        print(tag)
        ciphertext = connection.recv(1000)
        
        cipher_aes = AES.new(session_key, AES.MODE_EAX, nonce)
        data = cipher_aes.decrypt_and_verify(ciphertext, tag)

        print(data.decode("utf-8"))

Session key is  b'\xd4\x85J\xd0\x1b-k\xc9&^\xefM\x1e\x86\xc9\x8f'
b'\x07\xb3H,\x12\xa9\xdfFr\xecQ\x1a\xd5*:\x97'
b'"4|\x1a\xe5A^\xad}\x8d\xf1+g\x18\xa7\xff'
halo


KeyboardInterrupt: 